# **BERT FOR NER with label PER,ORG,MISC etc**

In [2]:
pip install transformers torch python-docx pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Colab / Python ready code
import os
import json
import pandas as pd
from docx import Document
from transformers import pipeline
from tqdm import tqdm
import torch

# ==========================
# CONFIG
# ==========================
INPUT_FOLDER = "/content/drive/MyDrive/research/DataSet"  # change path as needed
CSV_OUTPUT = "ner_results.csv"
JSON_OUTPUT = "ner_results.json"
MAX_CHARS = 800  # safe chunk length

# ==========================
# GPU / CPU Auto-detect
# ==========================
device = 0 if torch.cuda.is_available() else -1
print(f"Device set to: {'GPU' if device==0 else 'CPU'}")

# ==========================
# LOAD BERT NER MODEL
# ==========================
print("🔄 Loading BERT NER model...")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    device=device
)
print("✅ Model Loaded")

# ==========================
# READ DOCX FUNCTION
# ==========================
def read_docx(path):
    doc = Document(path)
    text = [para.text for para in doc.paragraphs if para.text.strip()]
    return " ".join(text)

# ==========================
# SPLIT LONG TEXT
# ==========================
def split_text(text, max_len=MAX_CHARS):
    return [text[i:i+max_len] for i in range(0, len(text), max_len)]

# ==========================
# MAIN PROCESS
# ==========================
results = []

# check if folder exists
if not os.path.exists(INPUT_FOLDER):
    raise FileNotFoundError(f"{INPUT_FOLDER} does not exist!")

files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".docx")]
if not files:
    raise FileNotFoundError(f"No .docx files found in {INPUT_FOLDER}")

for file in tqdm(files, desc="Processing DOCX files"):
    path = os.path.join(INPUT_FOLDER, file)
    text = read_docx(path)

    if not text.strip():
        continue

    chunks = split_text(text)
    for chunk in chunks:
        entities = ner(chunk)
        for ent in entities:
            results.append({
                "file": file,
                "entity": ent["word"],
                "label": ent["entity_group"],
                "confidence": float(round(ent["score"], 4))
            })

# ==========================
# SAVE OUTPUT
# ==========================
df = pd.DataFrame(results)
df.to_csv(CSV_OUTPUT, index=False)

with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\n🎉 NER COMPLETED")
print(f"📄 CSV: {CSV_OUTPUT}")
print(f"📄 JSON: {JSON_OUTPUT}")

# ==========================
# SAMPLE OUTPUT CHECK
# ==========================
print("\nFirst 5 rows of CSV:")
print(df.head())


Device set to: CPU
🔄 Loading BERT NER model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ Model Loaded


Processing DOCX files: 100%|██████████| 5/5 [08:13<00:00, 98.63s/it] 


🎉 NER COMPLETED
📄 CSV: ner_results.csv
📄 JSON: ner_results.json

First 5 rows of CSV:
                              file entity label  confidence
0  Process Management dataset.docx    POS  MISC      0.9425
1  Process Management dataset.docx   ##IX   ORG      0.5064
2  Process Management dataset.docx      E  MISC      0.5862
3  Process Management dataset.docx   ##LF   ORG      0.5245
4  Process Management dataset.docx   Java  MISC      0.9879


# **BERT with Defined Label**



```
# DOCX files (folder)
        ↓
BERT (pretrained NER)
        ↓
Keyword + pattern based mapping
        ↓
CONCEPT / TOPIC / METHOD / TECHNIQUE / OTHER
```


                    
A pre-trained BERT NER model was employed for entity extraction, followed by a rule-based classification layer to assign domain-specific labels such as Concept, Method, Technique, and Topic without additional training.

In [5]:
import os
import json
import pandas as pd
from docx import Document
from transformers import pipeline
from tqdm import tqdm
import torch

# ==========================
# CONFIG
# ==========================
INPUT_FOLDER = "/content/drive/MyDrive/research/DataSet"
CSV_OUTPUT = "custom_ner_results.csv"
JSON_OUTPUT = "custom_ner_results.json"
MAX_CHARS = 800

# ==========================
# DEVICE
# ==========================
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device==0 else 'CPU'}")

# ==========================
# LOAD BERT (NO TRAINING)
# ==========================
print("🔄 Loading BERT NER model...")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    device=device
)
print("✅ Model Loaded")

# ==========================
# CUSTOM LABEL MAPPER
# ==========================
def map_custom_label(word):
    w = word.lower()

    if any(k in w for k in ["algorithm", "model", "approach", "method"]):
        return "METHOD"

    if any(k in w for k in ["technique", "strategy", "procedure"]):
        return "TECHNIQUE"

    if any(k in w for k in ["concept", "theory", "principle", "abstraction", "encapsulation"]):
        return "CONCEPT"

    if any(k in w for k in ["introduction", "overview", "background", "topic"]):
        return "TOPIC"

    return "OTHER"

# ==========================
# READ DOCX (PAGE-WISE)
# ==========================
def read_docx_by_pages(path):
    doc = Document(path)
    pages, current = [], []

    for para in doc.paragraphs:
        if para.text.strip():
            current.append(para.text)
        else:
            if current:
                pages.append(" ".join(current))
                current = []

    if current:
        pages.append(" ".join(current))

    return pages

# ==========================
# SPLIT LONG TEXT
# ==========================
def split_text(text):
    return [text[i:i+MAX_CHARS] for i in range(0, len(text), MAX_CHARS)]

# ==========================
# MAIN PROCESS
# ==========================
results = []

files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".docx")]
if not files:
    raise FileNotFoundError("No DOCX files found")

for file in tqdm(files, desc="Processing DOCX files"):
    file_path = os.path.join(INPUT_FOLDER, file)
    pages = read_docx_by_pages(file_path)

    for page_no, page_text in enumerate(pages, start=1):
        if not page_text.strip():
            continue

        for chunk in split_text(page_text):
            entities = ner(chunk)

            for ent in entities:
                custom_label = map_custom_label(ent["word"])

                results.append({
                    "file": file,
                    "page": page_no,
                    "entity": ent["word"],
                    "label": custom_label,
                    "confidence": float(round(ent["score"], 4))
                })

# ==========================
# SAVE OUTPUT
# ==========================
df = pd.DataFrame(results)
df.to_csv(CSV_OUTPUT, index=False)

with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\n🎉 CUSTOM NER COMPLETED")
print(f"📄 CSV: {CSV_OUTPUT}")
print(f"📄 JSON: {JSON_OUTPUT}")
print("\nSample Output:")
print(df.head())


Device: CPU
🔄 Loading BERT NER model...


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


✅ Model Loaded


Processing DOCX files: 100%|██████████| 5/5 [09:00<00:00, 108.05s/it]


🎉 CUSTOM NER COMPLETED
📄 CSV: custom_ner_results.csv
📄 JSON: custom_ner_results.json

Sample Output:
                              file  page entity  label  confidence
0  Process Management dataset.docx     1    POS  OTHER      0.9425
1  Process Management dataset.docx     1   ##IX  OTHER      0.5064
2  Process Management dataset.docx     1      E  OTHER      0.5862
3  Process Management dataset.docx     1   ##LF  OTHER      0.5245
4  Process Management dataset.docx     1   Java  OTHER      0.9879


# **BERT + RuleBase**

In [6]:
# =====================================================
# INSTALL
# =====================================================
!pip install transformers python-docx pandas tqdm torch --quiet


# =====================================================
# IMPORTS
# =====================================================
import os
import json
import torch
import pandas as pd
from docx import Document
from transformers import pipeline
from tqdm import tqdm


# =====================================================
# CONFIG
# =====================================================
INPUT_FOLDER = "/content/drive/MyDrive/research/DataSet"  # DOCX folder
CSV_OUTPUT = "final_cs_domain_entities.csv"
JSON_OUTPUT = "final_cs_domain_entities.json"

MAX_CHARS = 800
CONF_THRESHOLD = 0.85   # strict confidence

DEVICE = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if DEVICE == 0 else "CPU")


# =====================================================
# LOAD PRETRAINED BERT NER (NO TRAINING)
# =====================================================
print("Loading BERT NER...")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
    device=DEVICE
)
print("Model Loaded ✅")


# =====================================================
# FILTERS & DICTIONARIES
# =====================================================
AUTHOR_NAMES = [
    "tanenbaum", "stallings", "silberschatz",
    "galvin", "gagne", "aho", "ullman"
]

GENERAL_SOFTWARE = [
    "firefox", "chrome", "edge", "google", "mozilla"
]

CONCEPT_TERMS = [
    "process", "thread", "deadlock", "critical section",
    "virtual memory", "abstraction", "inheritance",
    "encapsulation", "polymorphism", "normalization"
]

METHOD_TERMS = [
    "fcfs", "sjf", "lru", "fifo", "round robin",
    "dijkstra", "bellman ford", "kmp"
]

TECHNIQUE_TERMS = [
    "paging", "segmentation", "swapping",
    "deadlock avoidance", "deadlock detection"
]

TOPIC_TERMS = [
    "operating system", "computer networks",
    "database systems", "software engineering",
    "machine learning", "cloud computing"
]


# =====================================================
# HELPER FUNCTIONS
# =====================================================
def is_valid_entity(word):
    w = word.lower()
    if len(w) <= 3:
        return False
    if w in ["pos", "fig", "table", "example"]:
        return False
    if not w.replace(" ", "").isalpha():
        return False
    return True


def is_author(word):
    return word.lower() in AUTHOR_NAMES


def map_custom_label(word, context):
    w = word.lower()
    c = context.lower()

    if w in TOPIC_TERMS:
        return "TOPIC"

    if w in CONCEPT_TERMS:
        return "CONCEPT"

    if w in METHOD_TERMS or "algorithm" in c or "scheduler" in c:
        return "METHOD"

    if w in TECHNIQUE_TERMS or "technique" in c or "approach" in c:
        return "TECHNIQUE"

    if "is a" in c or "refers to" in c:
        return "CONCEPT"

    return "OTHER"


# =====================================================
# READ DOCX PAGE-WISE
# =====================================================
def read_docx_by_pages(path):
    doc = Document(path)
    pages = []
    current = []

    for para in doc.paragraphs:
        if para.text.strip():
            current.append(para.text)
        else:
            if current:
                pages.append(" ".join(current))
                current = []

    if current:
        pages.append(" ".join(current))

    return pages


def split_text(text):
    return [text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)]


# =====================================================
# MAIN PROCESS
# =====================================================
results = []

files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".docx")]
if not files:
    raise FileNotFoundError("No DOCX files found!")

for file in tqdm(files, desc="Processing DOCX"):
    pages = read_docx_by_pages(os.path.join(INPUT_FOLDER, file))

    for page_no, page_text in enumerate(pages, start=1):
        if not page_text.strip():
            continue

        for chunk in split_text(page_text):
            entities = ner(chunk)

            for ent in entities:
                word = ent["word"]
                score = ent["score"]

                if score < CONF_THRESHOLD:
                    continue
                if not is_valid_entity(word):
                    continue
                if is_author(word):
                    continue
                if word.lower() in GENERAL_SOFTWARE:
                    continue

                final_label = map_custom_label(word, chunk)

                results.append({
                    "file": file,
                    "page": page_no,
                    "entity": word,
                    "label": final_label,
                    "confidence": round(float(score), 4)
                })


# =====================================================
# SAVE OUTPUT
# =====================================================
df = pd.DataFrame(results)
df.to_csv(CSV_OUTPUT, index=False)

with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print("\n✅ NER COMPLETED (NO TRAINING)")
print(df.head(10))


Device: CPU
Loading BERT NER...


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Model Loaded ✅


Processing DOCX: 100%|██████████| 5/5 [08:38<00:00, 103.65s/it]


✅ NER COMPLETED (NO TRAINING)
                              file  page   entity    label  confidence
0  Process Management dataset.docx     1     Java    OTHER      0.9879
1  Process Management dataset.docx     4    MINIX    OTHER      0.9445
2  Process Management dataset.docx     5  Windows  CONCEPT      0.8907
3  Process Management dataset.docx     5     UNIX  CONCEPT      0.8611
4  Process Management dataset.docx     5     UNIX  CONCEPT      0.9425
5  Process Management dataset.docx     7  Windows    OTHER      0.9409
6  Process Management dataset.docx     8     UNIX  CONCEPT      0.8998
7  Process Management dataset.docx    10     Bach   METHOD      0.9995
8  Process Management dataset.docx    10     UNIX   METHOD      0.9774
9  Process Management dataset.docx    13  Windows  CONCEPT      0.9177


## **Graph Representation with Model Comparison**

In [ ]:
import json
import pandas as pd
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# =========================
# FILE PATHS
# =========================
GT_FILE = "/content/OS-Annotation(combine).txt"
PRED_FILE = "/content/final_custom_ner_results.csv"

# =========================
# LOAD GROUND TRUTH
# =========================
with open(GT_FILE, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# Convert GT → {(file, entity): label}
gt_map = {}
for doc in gt_data:
    file = doc["file"]
    for ent in doc["entities"]:
        key = (file.lower(), ent["entity"].lower())
        gt_map[key] = ent["label"]

# =========================
# LOAD PREDICTIONS
# =========================
pred_df = pd.read_csv(PRED_FILE)

pred_map = {}
for _, row in pred_df.iterrows():
    key = (row["file_name"].lower(), str(row["entity"]).lower())
    pred_map[key] = row["label"]

# =========================
# STRICT MATCHING
# =========================
y_true = []
y_pred = []

all_keys = set(gt_map.keys()).union(set(pred_map.keys()))

for key in all_keys:
    gt_label = gt_map.get(key, "O")       # O = not predicted
    pred_label = pred_map.get(key, "O")

    y_true.append(gt_label)
    y_pred.append(pred_label)

# =========================
# METRICS (MICRO)
# =========================
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="micro", zero_division=0
)

accuracy = accuracy_score(y_true, y_pred)

print("\n📊 STRICT EVALUATION (MICRO AVERAGE)")
print("-----------------------------------")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"Accuracy  : {accuracy:.4f}")

# =========================
# SAVE COMPARISON OUTPUT
# =========================
comparison = defaultdict(list)

for key in all_keys:
    file, entity = key
    comparison[file].append({
        "entity": entity,
        "ground_truth": gt_map.get(key, "O"),
        "prediction": pred_map.get(key, "O"),
        "match": gt_map.get(key, "O") == pred_map.get(key, "O")
    })

final_output = []
for file, entities in comparison.items():
    final_output.append({
        "file": file,
        "entities": entities
    })

with open("evaluation_comparison.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4)

print("\n📁 evaluation_comparison.json saved")


# **Bert docx formating**

In [ ]:
import json
from collections import defaultdict

# ---------- Input JSON ----------
INPUT_FILE = "/content/final_custom_ner_results.json"    # aapka JSON array file
OUTPUT_FILE = "bert_model_output.json"   # converted OS-Annotation format

# Load flat JSON
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------- Convert to OS-Annotation style ----------
os_annotation = defaultdict(list)

for item in data:
    file_name = item.get("file_name", "unknown").lower()
    entity_text = item.get("entity", "").strip()
    label = item.get("label", "OTHER").upper()

    os_annotation[file_name].append({
        "entity": entity_text,
        "label": label
    })

# Create final list
final_output = [{"file": f, "entities": ents} for f, ents in os_annotation.items()]

# Save JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4)

print(f"✅ Converted to OS-Annotation format: {OUTPUT_FILE}")
